Scratch work to find an efficient way to generate a complete set of bases.


This will be useful for defining larger ensembles, as the size of the space will scale as 2**n for n qubits in the ensemble.

In [1]:
import torch
from torch import nn

In [6]:
def define_basis(num_qubits):
    initial_basis = torch.arange(2).reshape((-1, 1))
    for i in range(num_qubits-1):
        copy_basis = initial_basis.clone()
        new_zeros = torch.zeros((copy_basis.shape[0], 1))
        new_ones = torch.ones((copy_basis.shape[0], 1))
        new_order_of_magnitude = torch.cat((new_zeros, new_ones), dim=0)
        new_basis = torch.cat((initial_basis, copy_basis), dim=0)
        new_basis = torch.cat((new_order_of_magnitude, new_basis), dim=1)
        initial_basis = new_basis
    return initial_basis.type(torch.uint8)

In [7]:
basis = define_basis(num_qubits=4)
print(basis.shape)
print(basis)

torch.Size([16, 4])
tensor([[0, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 1, 1],
        [0, 1, 0, 0],
        [0, 1, 0, 1],
        [0, 1, 1, 0],
        [0, 1, 1, 1],
        [1, 0, 0, 0],
        [1, 0, 0, 1],
        [1, 0, 1, 0],
        [1, 0, 1, 1],
        [1, 1, 0, 0],
        [1, 1, 0, 1],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.uint8)


## Controlled Gates Demo (CX, CH, CY, CZ)

Demonstrate the new controlled gate implementations and Bell state preparation.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import torch
from src import BaseQuantumCircuit, Zero, H, CX, CH, CY, CZ

# Bell state: H on qubit 0, then CX(0->1)
bell = BaseQuantumCircuit()
bell.add_qubit(Zero())
bell.add_qubit(Zero())
bell.add_gate(H(), [0])
bell.add_gate(CX(), [0, 1])
state = bell.forward()
probs = torch.real(state * state.conj()).squeeze()
print('Bell state probabilities:', probs.tolist())
print('Expected: [0.5, 0.0, 0.0, 0.5]')

# Controlled-Hadamard on |10>
ch = CH()
state10 = torch.zeros(1, 4, 1, dtype=torch.complex64)
state10[0, 2, 0] = 1.0
result = torch.matmul(ch.gate, state10)
print('\nCH|10> =', result.squeeze().tolist())

# Controlled-Y on |11>
cy = CY()
state11 = torch.zeros(1, 4, 1, dtype=torch.complex64)
state11[0, 3, 0] = 1.0
result = torch.matmul(cy.gate, state11)
print('CY|11> =', result.squeeze().tolist())

# Controlled-Z on |11>
cz = CZ()
state11 = torch.zeros(1, 4, 1, dtype=torch.complex64)
state11[0, 3, 0] = 1.0
result = torch.matmul(cz.gate, state11)
print('CZ|11> =', result.squeeze().tolist())